In [4]:
# IPython helpers to compare runs and pick the best "max messages" setting

import os, glob, json, math
from pathlib import Path
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- IO helpers ----------

def _read_any_yaml_json(path: Path) -> dict:
    """Best-effort read of a manifest/config (json or yaml)."""
    if not path.exists():
        return {}
    try:
        if path.suffix.lower() == ".json":
            return json.loads(path.read_text())
        import yaml  # pyyaml required
        return yaml.safe_load(path.read_text()) or {}
    except Exception:
        return {}

def load_parquet_glob(parquet_glob_or_dir: str,
                      needed_cols: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Load multiple parquet batches. Accepts a glob "*.parquet" or a directory.
    Returns a concatenated DataFrame (can be large).
    """
    p = Path(parquet_glob_or_dir)
    if p.is_dir():
        files = sorted(str(x) for x in p.glob("*.parquet"))
    else:
        files = sorted(glob.glob(parquet_glob_or_dir))

    if not files:
        raise FileNotFoundError(f"No parquet files match: {parquet_glob_or_dir}")

    cols = set(needed_cols or [])
    dfs = []
    for f in files:
        try:
            if cols:
                import pyarrow.parquet as pq
                have = set(pq.read_schema(f).names)
                use = list(cols & have)
                if not use:
                    continue
                df = pd.read_parquet(f, columns=use)
            else:
                df = pd.read_parquet(f)
            if not df.empty:
                dfs.append(df)
        except Exception as e:
            print(f"[WARN] failed to read {os.path.basename(f)}: {e}")
    if not dfs:
        raise RuntimeError("No parquet files could be read.")
    return pd.concat(dfs, ignore_index=True)

# ---------- Feature builders ----------

def _coerce_numeric(df, cols: List[str]):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

def compute_latencies(df: pd.DataFrame) -> Dict[str, pd.Series]:
    """
    Build the three latency series (in seconds) from consumer parquet batches.
    """
    out = {}
    _coerce_numeric(df, [
        "producer_timestamp","consumer_receive_timestamp",
        "application_timestamp","application_latency_seconds",
        "end_to_end_latency_seconds"
    ])

    # producer_consumer
    if {"producer_timestamp","consumer_receive_timestamp"}.issubset(df.columns):
        pc = (df["consumer_receive_timestamp"] - df["producer_timestamp"]).dropna()
        out["producer_consumer"] = pc

    # consumer_application
    if "application_latency_seconds" in df.columns:
        ca = pd.to_numeric(df["application_latency_seconds"], errors="coerce").dropna()
        out["consumer_application"] = ca
    elif {"application_timestamp","consumer_receive_timestamp"}.issubset(df.columns):
        ca = (df["application_timestamp"] - df["consumer_receive_timestamp"]).dropna()
        out["consumer_application"] = ca

    # producer_application (end-to-end)
    if "end_to_end_latency_seconds" in df.columns:
        e2e = pd.to_numeric(df["end_to_end_latency_seconds"], errors="coerce").dropna()
        out["producer_application"] = e2e
    elif {"application_timestamp","producer_timestamp"}.issubset(df.columns):
        e2e = (df["application_timestamp"] - df["producer_timestamp"]).dropna()
        out["producer_application"] = e2e

    return out

def clip_clean(v: pd.Series, cap_max_s: Optional[float]) -> pd.Series:
    v = v.replace([np.inf, -np.inf], np.nan).dropna()
    v = v[v >= 0]
    if cap_max_s is not None:
        v = v[v <= cap_max_s]
    return v

def frac_over_L(v: pd.Series, L: float) -> float:
    if len(v) == 0: return np.nan
    return float((v > L).mean())

def q_dict(v: pd.Series, qs=(0.50,0.90,0.95,0.99)) -> Dict[str, float]:
    if len(v) == 0:
        return {f"p{int(q*100)}": np.nan for q in qs}
    arr = np.quantile(v.to_numpy(), qs)
    return {f"p{int(q*100)}": float(val) for q, val in zip(qs, arr)}

def basic_health_grouped(df: pd.DataFrame) -> Dict[str, float]:
    """
    Compute duplicate & out-of-order rates correctly:
    - group by (topic, partition)
    - sort by offset within each group
    - duplicates: duplicated offsets within the same (topic, partition)
    - out-of-order: negative diffs of offsets within the group
    """
    h = {"out_of_order_rate": np.nan, "duplicate_rate": np.nan,
         "throughput_MBps": np.nan, "duration_s": np.nan}

    needed = {"topic","partition","offset"}
    if needed.issubset(df.columns):
        rates_ooo, rates_dup = [], []
        for (_, _), g in df.groupby(["topic","partition"], dropna=False):
            gg = g.sort_values("offset")
            off = pd.to_numeric(gg["offset"], errors="coerce")
            dup = off.duplicated(keep="first")
            rates_dup.append(float(dup.mean()))
            if len(off) > 1:
                ooo = (off.values[1:] < off.values[:-1])
                rates_ooo.append(float(ooo.mean()))
            else:
                rates_ooo.append(0.0)
        if rates_ooo:
            h["out_of_order_rate"] = float(np.nanmean(rates_ooo))
        if rates_dup:
            h["duplicate_rate"] = float(np.nanmean(rates_dup))

    # throughput from size_bytes + time span (best-effort)
    tcols = [c for c in ("producer_timestamp","consumer_receive_timestamp") if c in df.columns]
    if tcols:
        t = pd.to_numeric(df[tcols[0]], errors="coerce").dropna()
        if len(t) >= 2:
            dur = float(t.max() - t.min())
            h["duration_s"] = dur if dur > 0 else np.nan
            if "size_bytes" in df.columns:
                sz = pd.to_numeric(df["size_bytes"], errors="coerce").dropna()
                bytes_total = float(sz.sum()) if len(sz) else np.nan
                if bytes_total and dur and dur > 0:
                    h["throughput_MBps"] = bytes_total / (1024*1024) / dur
    return h

# ---------- Block maxima + config helpers ----------

def read_block_maxima(blockmax_csv: str) -> pd.DataFrame:
    if not Path(blockmax_csv).exists():
        raise FileNotFoundError(blockmax_csv)
    return pd.read_csv(blockmax_csv)

def extract_run_config(metrics_dir: Path) -> dict:
    """
    Try to find run manifest or config in the same folder as block_maxima.csv
    and pull relevant knobs (e.g., MAX_MESSAGES).
    """
    cfg = {}
    for name in ("run_manifest.json","run_manifest.yaml","run_manifest.yml"):
        d = _read_any_yaml_json(metrics_dir / name)
        if d: cfg.update(d if isinstance(d, dict) else {"manifest": d})
    for f in metrics_dir.glob("pipeline-configmap*.yaml"):
        d = _read_any_yaml_json(f)
        if d: cfg.update({"pipeline_config": d})
    return cfg

def read_max_messages(metrics_dir: str) -> Optional[int]:
    """Find MAX_MESSAGES in pipeline-configmap snapshot or run manifest."""
    import yaml
    # 1) pipeline-configmap snapshot
    for yml in glob.glob(os.path.join(metrics_dir, "pipeline-configmap_cfg*.yaml")):
        try:
            with open(yml, "r") as f:
                cfg = yaml.safe_load(f) or {}
            data = cfg.get("data", {}) or {}
            v = data.get("MAX_MESSAGES")
            if v is not None:
                return int(v)
        except Exception:
            pass
    # 2) run manifest
    for name in ("run_manifest.json","run_manifest.yaml","run_manifest.yml"):
        p = os.path.join(metrics_dir, name)
        if not os.path.exists(p): 
            continue
        try:
            with open(p, "r") as f:
                cfg = json.load(f) if p.endswith(".json") else yaml.safe_load(f)
            data = (cfg.get("data", {}) or {})
            for k in ("MAX_MESSAGES","max_messages","maxMsg"):
                v = data.get(k) or cfg.get(k)
                if v is not None:
                    return int(v)
        except Exception:
            pass
    return None

# ---------- Per-run analysis ----------

def analyze_run(parquet_glob_or_dir: str,
                blockmax_csv: str,
                label: str,
                Lpc: float = 0.25,    # seconds
                Le2e: float = 0.50,   # seconds
                Lca: float = 0.05,    # seconds
                slo_pc: float = 0.01, slo_e2e: float = 0.01, slo_ca: float = 0.01,
                cap_max_s: Optional[float] = 1.0) -> dict:
    """
    Analyze one run and return a rich dict of summary metrics + pass/fail vs SLO.
    """
    need = [
        "index","producer_timestamp","consumer_receive_timestamp",
        "application_timestamp","application_latency_seconds","end_to_end_latency_seconds",
        "size_bytes","target_rate","topic","partition","offset",
        "producer_pod","consumer_pod"
    ]
    df = load_parquet_glob(parquet_glob_or_dir, needed_cols=need)

    # Latencies
    lats = compute_latencies(df)
    pc = clip_clean(lats.get("producer_consumer", pd.Series([], dtype=float)), cap_max_s)
    ca = clip_clean(lats.get("consumer_application", pd.Series([], dtype=float)), cap_max_s)
    e2e= clip_clean(lats.get("producer_application", pd.Series([], dtype=float)), cap_max_s)

    # Tail fractions
    f_pc  = frac_over_L(pc, Lpc)
    f_ca  = frac_over_L(ca, Lca)
    f_e2e = frac_over_L(e2e, Le2e)

    # Percentiles
    q_pc  = q_dict(pc)
    q_ca  = q_dict(ca)
    q_e2e = q_dict(e2e)

    # Health
    h = basic_health_grouped(df)

    # Blocks info
    bdf = read_block_maxima(blockmax_csv)
    n_blocks_total = int(len(bdf))
    n_blocks_by_metric = bdf.groupby("metric")["block_max_value"].size().to_dict()

    # Config snapshot
    metrics_dir = Path(blockmax_csv).parent
    run_cfg = extract_run_config(metrics_dir)
    max_messages = read_max_messages(str(metrics_dir))

    summary = {
        "label": label,
        "paths": {
            "parquet": parquet_glob_or_dir,
            "blockmax": blockmax_csv,
            "metrics_dir": str(metrics_dir)
        },
        "config": {"max_messages": max_messages},
        "health": h,
        "blocks": {
            "n_total": n_blocks_total,
            "by_metric": n_blocks_by_metric
        },
        "producer_consumer": {
            "n": int(pc.size), "L": Lpc, "frac_over_L": f_pc, **q_pc,
            "slo": slo_pc, "pass": (f_pc <= slo_pc) if not math.isnan(f_pc) else False
        },
        "consumer_application": {
            "n": int(ca.size), "L": Lca, "frac_over_L": f_ca, **q_ca,
            "slo": slo_ca, "pass": (f_ca <= slo_ca) if not math.isnan(f_ca) else False
        },
        "producer_application": {
            "n": int(e2e.size), "L": Le2e, "frac_over_L": f_e2e, **q_e2e,
            "slo": slo_e2e, "pass": (f_e2e <= slo_e2e) if not math.isnan(f_e2e) else False
        },
    }
    # score: more SLO passes is better; then lower e2e p99; then lower dup/ooo
    passes = int(summary["producer_consumer"]["pass"]) + int(summary["consumer_application"]["pass"]) + int(summary["producer_application"]["pass"])
    p99_e2e = summary["producer_application"]["p99"]
    score = (
        passes,
        -p99_e2e if not math.isnan(p99_e2e) else float("-inf"),
        -(summary["health"]["duplicate_rate"] if not math.isnan(summary["health"]["duplicate_rate"]) else 0.0),
        -(summary["health"]["out_of_order_rate"] if not math.isnan(summary["health"]["out_of_order_rate"]) else 0.0),
    )
    summary["_score_tuple"] = score
    summary["_passes"] = passes
    return summary

def flatten(run_summary: dict) -> dict:
    """Flatten one run summary into a single-row dict (easy to tabulate)."""
    return {
        "label": run_summary["label"],
        "parquet": run_summary["paths"]["parquet"],
        "blockmax": run_summary["paths"]["blockmax"],
        "metrics_dir": run_summary["paths"]["metrics_dir"],
        "max_messages": run_summary["config"]["max_messages"],
        "blocks_total": run_summary["blocks"]["n_total"],
        "pc_n": run_summary["producer_consumer"]["n"],
        "pc_L": run_summary["producer_consumer"]["L"],
        "pc_frac_over_L": run_summary["producer_consumer"]["frac_over_L"],
        "pc_p99": run_summary["producer_consumer"]["p99"],
        "pc_pass": run_summary["producer_consumer"]["pass"],
        "ca_n": run_summary["consumer_application"]["n"],
        "ca_L": run_summary["consumer_application"]["L"],
        "ca_frac_over_L": run_summary["consumer_application"]["frac_over_L"],
        "ca_p99": run_summary["consumer_application"]["p99"],
        "ca_pass": run_summary["consumer_application"]["pass"],
        "e2e_n": run_summary["producer_application"]["n"],
        "e2e_L": run_summary["producer_application"]["L"],
        "e2e_frac_over_L": run_summary["producer_application"]["frac_over_L"],
        "e2e_p99": run_summary["producer_application"]["p99"],
        "e2e_pass": run_summary["producer_application"]["pass"],
        "throughput_MBps": run_summary["health"]["throughput_MBps"],
        "duplicate_rate": run_summary["health"]["duplicate_rate"],
        "out_of_order_rate": run_summary["health"]["out_of_order_rate"],
        "passes_total": run_summary["_passes"],
    }


In [5]:
def choose_best(run_summaries: List[dict]) -> dict:
    """Pick the run with the highest pass count; then lowest e2e p99; then lowest dup/ooo."""
    if not run_summaries:
        raise ValueError("No runs given.")
    return max(run_summaries, key=lambda r: r["_score_tuple"])

# ---------- YOUR RUN PATHS ----------


    '''
    dict(
        label="run_20250909_2326",
        parquet="../20250909_232655/consumer/consumer-sts-0_consumer-result/processed/2025-09-09/batch_consumer-sts-*.parquet",
        blockmax="../20250909_232655/merge/merge-sts-0_merge-metrics/2025-09-09_20-04-56/block_maxima.csv",
    ),
    
    dict(
        label="run_20250910_1436",
        parquet="../20250910_143637/consumer/consumer-sts-0_consumer-result/processed/2025-09-10/batch_consumer-sts-*.parquet",
        blockmax="../20250910_143637/merge/merge-sts-0_merge-metrics/2025-09-10_00-37-32/block_maxima.csv",
    ),
    dict(
        label="run_20250910_2034",
        parquet="../20250910_203447/consumer/consumer-sts-0_consumer-result/2025-09-10_18-34/batch_consumer-sts-*.parquet",
        blockmax="../20250910_203447/merge/merge-sts-0_merge-metrics/2025-09-10_18-34-48/block_maxima.csv",
    ),
    
    dict(
        label="run_20250910-2016",
        parquet="../20250910_2016/2025-09-10_18-16/batch_consumer-sts-*.parquet",
        blockmax="../20250910_2016/2025-09-10_18-16-41/block_maxima.csv",
    ),
    dict(
        label="run_20250910-2109",
        parquet="../20250910_210900/consumer/consumer-sts-0_consumer-result/processed/2025-09-10/batch_consumer-sts-*.parquet",
        blockmax="../20250910_210900/merge/merge-sts-0_merge-metrics/2025-09-10_20-39-35/block_maxima.csv",
    ),
    dict(
        label="run_20250910-2143",
        parquet="../20250910_214347/consumer/consumer-sts-0_consumer-result/processed/2025-09-10/batch_consumer-sts-*.parquet",
        blockmax="../20250910_214347/merge/merge-sts-0_merge-metrics/2025-09-10_21-10-23/block_maxima.csv",
    ),
    '''
R = [
    dict
    ( 
      label="run_20250913-1545",
      parquet="../20250913_154558/consumer/consumer-sts-0_consumer-result/processed/2025-09-13/batch_consumer-sts-*.parquet",
      blockmax="../20250913_154558/merge/merge-sts-0_merge-metrics/2025-09-13_00-49-05/block_maxima.csv",  
    )
]

# ---------- Thresholds (seconds) and SLOs (fraction allowed over L) ----------

Lpc, Le2e, Lca = 0.25, 0.50, 0.05
slo_pc = slo_e2e = slo_ca = 0.01
cap_max_s = 5.0   # drop obviously bogus >5s values from synthetic runs; set to None to disable trimming

# ---------- Analyze all runs ----------

summaries = []
for r in R:
    s = analyze_run(
        parquet_glob_or_dir=r["parquet"],
        blockmax_csv=r["blockmax"],
        label=r["label"],
        Lpc=Lpc, Le2e=Le2e, Lca=Lca,
        slo_pc=slo_pc, slo_e2e=slo_e2e, slo_ca=slo_ca,
        cap_max_s=cap_max_s
    )
    summaries.append(s)

best = choose_best(summaries)

# Build table
rows = [flatten(x) for x in summaries]
table = pd.DataFrame(rows)

# Fill/repair max_messages from metrics_dir when missing
def _fill_max_messages(row):
    v = row.get("max_messages", None)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return read_max_messages(row["metrics_dir"])
    return v

table["max_messages"] = table.apply(_fill_max_messages, axis=1)

# Order nicely
table = table.sort_values(["passes_total","e2e_p99"], ascending=[False, True]).reset_index(drop=True)

print(f"Best run by score: {best['label']}")
display(table)
# ---------- Compare by MAX_MESSAGES across runs ----------

if "max_messages" in table.columns and table["max_messages"].notna().any():
    by_mm = (
        table.groupby("max_messages", dropna=False)
             .agg(
                 runs=("label","count"),
                 passes_total=("passes_total","sum"),
                 e2e_p99_median=("e2e_p99","median"),
                 pc_frac_over_L_median=("pc_frac_over_L","median"),
                 ca_frac_over_L_median=("ca_frac_over_L","median"),
                 e2e_frac_over_L_median=("e2e_frac_over_L","median"),
                 duplicate_rate_mean=("duplicate_rate","mean"),
                 out_of_order_rate_mean=("out_of_order_rate","mean"),
             )
             .sort_values(["max_messages","e2e_p99_median"], ascending=[False, True])
             .reset_index()
    )
    print("\nAggregate comparison by MAX_MESSAGES:")
    display(by_mm)
    if not by_mm.empty and pd.notna(by_mm.loc[0, "max_messages"]):
        print(f"\nRecommended MAX_MESSAGES (Phase A): {int(by_mm.loc[0, 'max_messages'])}")
else:
    print("\n(max_messages not found in snapshots/manifests; ensure merge saved pipeline-configmap_cfg*.yaml or run_manifest.*)")


Best run by score: run_20250913-1545


,label,parquet,blockmax,metrics_dir,max_messages,blocks_total,pc_n,pc_L,pc_frac_over_L,pc_p99,...,ca_pass,e2e_n,e2e_L,e2e_frac_over_L,e2e_p99,e2e_pass,throughput_MBps,duplicate_rate,out_of_order_rate,passes_total
0,run_20250913-1545,../20250913_154558/consumer/consumer-sts-0_con...,../20250913_154558/merge/merge-sts-0_merge-met...,../20250913_154558/merge/merge-sts-0_merge-met...,5000,13746,22905672,0.25,0.747508,0.714315,...,True,22905658,0.5,0.284547,0.729362,False,13.352149,0.0,0.0,1



Aggregate comparison by MAX_MESSAGES:


,max_messages,runs,passes_total,e2e_p99_median,pc_frac_over_L_median,ca_frac_over_L_median,e2e_frac_over_L_median,duplicate_rate_mean,out_of_order_rate_mean
0,5000,1,1,0.729362,0.747508,0.0,0.284547,0.0,0.0



Recommended MAX_MESSAGES (Phase A): 5000


### out_csv = "../plotsmax_message_comparison.csv"
table.to_csv(out_csv, index=False)
